# =========================================================
# AEGIS ZERO-DAY WINDOWS LOG ANOMALY DETECTION
# Isolation Forest Based Detection Pipeline
# =========================================================

In [40]:
import os
import re

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

import joblib

In [ ]:
LOG_FILE = r"../../Dataset/Zero_Day/Windows.log"


In [42]:
print(os.path.exists(LOG_FILE))

True


In [43]:
with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:

    for i in range(20):
        print(f.readline())
        

2016-09-28 04:30:30, Info                  CBS    Starting TrustedInstaller initialization.

2016-09-28 04:30:30, Info                  CBS    Loaded Servicing Stack v6.1.7601.23505 with Core: C:\Windows\winsxs\amd64_microsoft-windows-servicingstack_31bf3856ad364e35_6.1.7601.23505_none_681aa442f6fed7f0\cbscore.dll

2016-09-28 04:30:31, Info                  CSI    00000001@2016/9/27:20:30:31.455 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fef9fb9b6d @0x7fef9f8358f @0xff83e97c @0xff83d799 @0xff83db2f)

2016-09-28 04:30:31, Info                  CSI    00000002@2016/9/27:20:30:31.458 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fefa006ade @0x7fef9fd2984 @0x7fef9f83665 @0xff83e97c @0xff83d799)

2016-09-28 04:30:31, Info                  CSI    00000003@2016/9/27:20:30:31.458 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fefa1c8728 @0x7fefa1c8856 @0xff83e474 @0xff83d7de @0xff83db2f)

2016-09-28 04:30:31, In

In [44]:
parsed_logs = []

In [45]:
eventid_pattern = r'EventID[:=]\s*(\d+)'

ip_pattern = r'(\d+\.\d+\.\d+\.\d+)'

process_pattern = r'([A-Za-z0-9_\-]+\.exe)'

user_pattern = r'User[:=]\s*([A-Za-z0-9_\-]+)'

In [46]:
MAX_LINES = 500000

In [47]:
print(f"Reading first {MAX_LINES} logs...")

Reading first 500000 logs...


In [48]:
with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:

    for idx, line in enumerate(tqdm(f)):

        if idx >= MAX_LINES:
            break

        eventid = re.search(eventid_pattern, line)
        ip = re.search(ip_pattern, line)
        process = re.search(process_pattern, line)
        user = re.search(user_pattern, line)

        parsed_logs.append({

            "event_id": eventid.group(1) if eventid else "0",

            "ip": ip.group(1) if ip else "0.0.0.0",

            "process": process.group(1) if process else "unknown.exe",

            "user": user.group(1) if user else "unknown",

            "raw_log": line[:300]
        })

500000it [00:06, 75230.41it/s]


In [49]:
df = pd.DataFrame(parsed_logs)

df.head()

,event_id,ip,process,user,raw_log
0,0,0.0.0.0,unknown.exe,unknown,"﻿2016-09-28 04:30:30, Info CB..."
1,0,6.1.7601.23505,unknown.exe,unknown,"2016-09-28 04:30:30, Info CBS..."
2,0,0.0.0.6,unknown.exe,unknown,"2016-09-28 04:30:31, Info CSI..."
3,0,0.0.0.6,unknown.exe,unknown,"2016-09-28 04:30:31, Info CSI..."
4,0,0.0.0.6,unknown.exe,unknown,"2016-09-28 04:30:31, Info CSI..."


In [50]:
# =========================================
# CHECK NULL VALUES
# =========================================

print(df.isnull().sum())

event_id    0
ip          0
process     0
user        0
raw_log     0
dtype: int64


In [51]:
print(df.shape)

(500000, 5)


In [52]:
# =========================================
# CHECK UNIQUE VALUES
# =========================================

print("\nUnique Event IDs:")
print(df["event_id"].nunique())

print("\nUnique Processes:")
print(df["process"].nunique())

print("\nUnique Users:")
print(df["user"].nunique())


Unique Event IDs:
1

Unique Processes:
2

Unique Users:
1


In [53]:
event_encoder = LabelEncoder()

df["event_id_encoded"] = event_encoder.fit_transform(
    df["event_id"]
)

In [54]:
process_encoder = LabelEncoder()

df["process_encoded"] = process_encoder.fit_transform(
    df["process"]
)

In [55]:
user_encoder = LabelEncoder()

df["user_encoded"] = user_encoder.fit_transform(
    df["user"]
)

In [56]:
df["ip_last_octet"] = df["ip"].apply(

    lambda x: int(x.split(".")[-1])

    if "." in x else 0
)

In [57]:
features = [

    "event_id_encoded",

    "process_encoded",

    "user_encoded",

    "ip_last_octet"
]

X = df[features]

X.head()

,event_id_encoded,process_encoded,user_encoded,ip_last_octet
0,0,1,0,0
1,0,1,0,23505
2,0,1,0,6
3,0,1,0,6
4,0,1,0,6


In [58]:
imputer = SimpleImputer(strategy="mean")

X = imputer.fit_transform(X)

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(400000, 4)
(100000, 4)


In [60]:
model = IsolationForest(

    n_estimators=100,

    contamination=0.01,

    random_state=42,

    n_jobs=-1
)

In [61]:
print("\nSelected Features:")
print(features)


Selected Features:
['event_id_encoded', 'process_encoded', 'user_encoded', 'ip_last_octet']


In [62]:
model.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [63]:
# =========================================
# MODEL EVALUATION
# =========================================

train_preds = model.predict(X_train)

test_preds = model.predict(X_test)

train_anomaly_percent = (
    np.mean(train_preds == -1) * 100
)

test_anomaly_percent = (
    np.mean(test_preds == -1) * 100
)

print(f"Train anomaly %: {train_anomaly_percent:.2f}")

print(f"Test anomaly %: {test_anomaly_percent:.2f}")

Train anomaly %: 0.94
Test anomaly %: 0.92


In [64]:
test_predictions = model.predict(X_test)

test_scores = model.decision_function(X_test)

In [65]:
df["anomaly"] = model.predict(X)

In [66]:
df["anomaly_score"] = model.decision_function(X)

In [67]:
# =========================================
# ANOMALY SCORE STATISTICS
# =========================================

print(df["anomaly_score"].describe())

count    500000.000000
mean          0.301171
std           0.073780
min          -0.019695
25%           0.271388
50%           0.334637
75%           0.353870
max           0.353870
Name: anomaly_score, dtype: float64


In [68]:
anomalies = df[df["anomaly"] == -1]

print("Total Anomalies Found:", len(anomalies))

anomalies.head()

Total Anomalies Found: 4670


,event_id,ip,process,user,raw_log,event_id_encoded,process_encoded,user_encoded,ip_last_octet,anomaly,anomaly_score
454,0,7.1.7601.16492,unknown.exe,unknown,"2016-09-28 04:30:37, Info CBS...",0,1,0,16492,-1,-0.019695
474,0,6.3.9600.17681,unknown.exe,unknown,"2016-09-28 04:30:37, Info CBS...",0,1,0,17681,-1,-0.016673
498,0,7.2.7601.16415,unknown.exe,unknown,"2016-09-28 04:30:39, Info CBS...",0,1,0,16415,-1,-0.019153
584,0,6.1.1.11,unknown.exe,unknown,"2016-09-28 04:30:43, Info CBS...",0,1,0,11,-1,-0.004128
592,0,6.3.9600.17388,unknown.exe,unknown,"2016-09-28 04:30:43, Info CBS...",0,1,0,17388,-1,-0.017642


In [69]:
# =========================================
# TOP MOST SUSPICIOUS LOGS
# =========================================

most_suspicious = anomalies.sort_values(
    by="anomaly_score"
)

most_suspicious[[
    "raw_log",
    "anomaly_score"
]].head(20)

,raw_log,anomaly_score
272143,"2016-10-11 03:05:24, Info CBS...",-0.019695
272142,"2016-10-11 03:05:24, Info CBS...",-0.019695
272141,"2016-10-11 03:05:24, Info CBS...",-0.019695
272140,"2016-10-11 03:05:24, Info CBS...",-0.019695
3663,"2016-09-29 22:29:07, Info CBS...",-0.019695
3346,"2016-09-29 22:28:50, Info CBS...",-0.019695
2539,"2016-09-29 03:00:42, Info CBS...",-0.019695
272372,"2016-10-11 03:05:24, Info CBS...",-0.019695
272371,"2016-10-11 03:05:24, Info CBS...",-0.019695
272370,"2016-10-11 03:05:24, Info CBS...",-0.019695


In [70]:
most_suspicious = anomalies.sort_values(
    by="anomaly_score"
)

most_suspicious[[
    "raw_log",
    "anomaly_score"
]].head(20)

,raw_log,anomaly_score
272143,"2016-10-11 03:05:24, Info CBS...",-0.019695
272142,"2016-10-11 03:05:24, Info CBS...",-0.019695
272141,"2016-10-11 03:05:24, Info CBS...",-0.019695
272140,"2016-10-11 03:05:24, Info CBS...",-0.019695
3663,"2016-09-29 22:29:07, Info CBS...",-0.019695
3346,"2016-09-29 22:28:50, Info CBS...",-0.019695
2539,"2016-09-29 03:00:42, Info CBS...",-0.019695
272372,"2016-10-11 03:05:24, Info CBS...",-0.019695
272371,"2016-10-11 03:05:24, Info CBS...",-0.019695
272370,"2016-10-11 03:05:24, Info CBS...",-0.019695


In [71]:
# =========================================
# CREATE OUTPUT DIRECTORIES
# =========================================

os.makedirs(
    "../../trained_models/zero_day/outputs",
    exist_ok=True
)

In [72]:
joblib.dump(

    model,

    "../../trained_models/zero_day/aegis_zero_day_model.pkl"
)


['../../trained_models/zero_day/aegis_zero_day_model.pkl']

In [73]:
joblib.dump(

    event_encoder,

    "../../trained_models/zero_day/event_encoder.pkl"
)

joblib.dump(

    process_encoder,

    "../../trained_models/zero_day/process_encoder.pkl"
)

joblib.dump(

    user_encoder,

    "../../trained_models/zero_day/user_encoder.pkl"
)

['../../trained_models/zero_day/user_encoder.pkl']

In [74]:
df.to_parquet(

    "../../trained_models/zero_day/outputs/processed_logs.parquet",

    index=False
)

In [75]:
anomalies.to_csv(

    "../../trained_models/zero_day/outputs/zero_day_anomalies.csv",

    index=False
)

In [76]:
print("===================================")
print("AEGIS Zero-Day Model Training Done")
print("===================================")

print("Model Saved Successfully")
print("Anomalies Saved Successfully")
print("Processed Logs Saved Successfully")

AEGIS Zero-Day Model Training Done
Model Saved Successfully
Anomalies Saved Successfully
Processed Logs Saved Successfully
